# Module 04: Seaborn for Machine Learning
## Notebook 03: Matrix Plots and Correlation Heatmaps

Before feeding features into linear models, neural networks, or tree ensembles, identifying feature redundancy and multicollinearity is paramount. Seaborn's heatmap and clustermap tools provide intuitive matrix representations of feature associations.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Compute and interpret Pearson and Spearman correlation matrices (`df.corr()`).
2. Construct annotated correlation heatmaps with diverging colormaps.
3. Mask the upper triangle of heatmaps to eliminate visual redundancy.
4. Apply hierarchical agglomerative clustering using `sns.clustermap()`.
5. **Advanced:** Construct dual metadata color bars on clustermaps and automate multicollinearity screening using correlation thresholds.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
sns.set_theme(style="white")
print("Seaborn loaded!")

### 1. Generating Feature Correlation Matrices

Correlation measures the linear relationship between pairs of continuous variables:
- $+1.0$: Perfect positive correlation.
- $0.0$: No linear relationship.
- $-1.0$: Perfect negative correlation.

In [ ]:
# Generating synthetic medical diagnostic dataset
n_patients = 120
age = rng.normal(52, 12, size=n_patients)
bmi = rng.normal(27, 4, size=n_patients)
blood_pressure = age * 0.45 + bmi * 0.8 + rng.normal(70, 10, size=n_patients)
cholesterol = age * 0.55 + bmi * 0.6 + rng.normal(120, 15, size=n_patients)
glucose = bmi * 1.1 + rng.normal(85, 12, size=n_patients)
insulin = glucose * 0.75 + rng.normal(15, 5, size=n_patients)

df_medical = pd.DataFrame({
    'Age': age,
    'BMI': bmi,
    'Blood_Pressure': blood_pressure,
    'Cholesterol': cholesterol,
    'Glucose': glucose,
    'Insulin': insulin
})

corr_matrix = df_medical.corr()
print("Pearson Correlation Matrix:\n", corr_matrix.round(2))

---
### 2. Masked Correlation Heatmaps

Since correlation matrices are perfectly symmetric ($r_{ij} = r_{ji}$) and the diagonal is always $1.0$:
- Displaying both triangles creates redundant visual clutter.
- Using `np.triu()` masks the upper triangle, directing focus exclusively to informative unique pairs.

In [ ]:
# Create boolean mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(8, 6.5))

# Diverging palette centered at 0
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    vmax=1.0,
    vmin=-1.0,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation (r)"},
    ax=ax
)

ax.set_title("Triangular Masked Feature Correlation Heatmap", fontsize=13, fontweight='bold', pad=12)
plt.show()

---
### 3. Hierarchical Clustermaps: `sns.clustermap()`

While a standard heatmap displays features in their arbitrary dataset column order, `sns.clustermap()` applies **Agglomerative Hierarchical Clustering**:
- Computes pairwise distances between features.
- Reorders rows and columns so that correlated features are placed adjacent to one another.
- Displays a **Dendrogram tree** depicting the cluster hierarchy!

In [ ]:
# Clustermap of feature correlations
g = sns.clustermap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1.0,
    vmax=1.0,
    figsize=(7.5, 7.5),
    dendrogram_ratio=0.15,
    cbar_pos=(0.02, 0.8, 0.04, 0.15)
)

g.fig.suptitle("Hierarchically Clustered Feature Dendrogram", fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
### 4. Advanced Complex Usage: Clustermaps with Sample Class Metadata and Automated Collinearity Screening

In high-dimensional biology, genetics, or multi-sensor analytics:
1. **Sample Metadata Color Bars (`row_colors`):** Annotate patient risk cohorts (e.g. High Risk vs Low Risk) alongside unsupervised sample-level feature clustering to see if clusters naturally align with labels.
2. **Automated Collinearity Filtering Pipeline:** Programmatically detects and isolates redundant features above threshold $|r| > 0.70$ for downstream feature pruning.

In [ ]:
# 1. Clustermap with Row Metadata Colors
# Assign synthetic risk labels
risk_scores = df_medical['Blood_Pressure'] * 0.4 + df_medical['Glucose'] * 0.6
risk_labels = np.where(risk_scores > np.median(risk_scores), 'High_Risk', 'Low_Risk')
palette_meta = {'High_Risk': '#e74c3c', 'Low_Risk': '#3498db'}
row_colors = pd.Series(risk_labels).map(palette_meta)

g_meta = sns.clustermap(
    df_medical.iloc[:40],  # First 40 patients
    standard_scale=1,       # Z-score normalize features
    row_colors=row_colors[:40],
    cmap='coolwarm',
    figsize=(9, 7),
    cbar_kws={'label': 'Z-Score'}
)
g_meta.fig.suptitle("Patient Biomarker Clustermap with Diagnostic Risk Metadata", fontsize=12, fontweight='bold', y=1.02)
plt.show()

# 2. Automated Collinearity Pruning Pipeline
def prune_collinear_features(df_corr: pd.DataFrame, threshold: float = 0.70) -> list:
    # Programmatically identify features to prune due to excessive correlation
    upper_tri = df_corr.where(np.triu(np.ones(df_corr.shape), k=1).astype(bool))
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column].abs() > threshold)]
    return to_drop

redundant_features = prune_collinear_features(corr_matrix, threshold=0.70)
print(f"Collinear Features flagged for removal (|r| > 0.70): {redundant_features}")
clean_feature_subset = df_medical.drop(columns=redundant_features)
print(f"Remaining Non-Redundant Features: {list(clean_feature_subset.columns)}")

### Summary & Next Steps
In this notebook, you mastered:
- Computing Pearson vs Spearman correlation matrices.
- Eliminating visual clutter with upper-triangle masked heatmaps.
- Hierarchical feature clustering with `sns.clustermap()`.
- Metadata color annotations and automated collinearity screening pipelines.

**Next Notebook:** `04_multi_plot_grids_and_eda_workflow.ipynb` — Pair plots, Facet grids, and end-to-end Exploratory Data Analysis (EDA) pipelines.